# 📘 Week 8: Audio Denoising & Feature Extraction
Welcome to Week 8! In this module, we will explore **Audio Denoising via Spectral Subtraction** and extract advanced audio features like **Mel-Frequency Cepstral Coefficients (MFCCs)** and **Chroma** vectors.

## 🎯 Learning Objectives:
- Understand the mathematics and implementation of **Spectral Subtraction** for noise reduction.
- Reconstruct denoised audio signals using the **Inverse Short-Time Fourier Transform (iSTFT)**.
- Extract and interpret **MFCCs** for speech representation.
- Extract and interpret **Chroma** features for musical octave analysis.
- Apply basic audio effects like **Pitch Shifting** and **Time Stretching**.

## 1. Noise Simulation and the Short-Time Fourier Transform (STFT)
First, we will simulate a noisy signal by adding white Gaussian noise to a synthetic voice-like signal (a combination of sine waves). We will then compute the STFT of both clean and noisy signals.

Let's prepare our signals:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import librosa
import scipy.signal as signal
from IPython.display import Audio, display

fs = 16000  # 16 kHz sampling rate
t = np.linspace(0, 3.0, int(fs * 3.0), endpoint=False)

# Create synthetic voice-like signal (voiced vowels have rich harmonics)
clean_signal = np.sin(2 * np.pi * 150 * t) + 0.5 * np.sin(2 * np.pi * 300 * t) + 0.25 * np.sin(2 * np.pi * 450 * t)
# Add silence at the beginning (first 0.5 seconds) to simulate a noise-only segment
clean_signal[t < 0.5] = 0.0

# Add stationary white noise
noise_level = 0.3
noise = np.random.normal(0, noise_level, len(clean_signal))
noisy_signal = clean_signal + noise

# Compute STFT of noisy signal
n_fft = 512
hop_length = 128
win_length = 512

S_noisy = librosa.stft(noisy_signal, n_fft=n_fft, hop_length=hop_length, win_length=win_length, center=False)
S_noisy_mag, S_noisy_phase = librosa.magphase(S_noisy)

# Plot clean and noisy waveforms
plt.figure(figsize=(12, 5))
plt.subplot(2, 1, 1)
plt.plot(t, clean_signal, label="Clean Signal", color='g', alpha=0.7)
plt.title("Clean vs Noisy Signals in Time Domain")
plt.legend()
plt.grid(True)

plt.subplot(2, 1, 2)
plt.plot(t, noisy_signal, label="Noisy Signal", color='r', alpha=0.5)
plt.xlabel("Time (seconds)")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

## 2. Spectral Subtraction Denoising
**Spectral Subtraction** is an intuitive algorithm based on the assumption that noise is additive and stationary:
$$y(t) = x(t) + n(t)$$
In the STFT magnitude spectrum domain:
$$\hat{|X(f, t)|} = \max(|Y(f, t)| - \alpha \cdot \mu(f), \beta \cdot |Y(f, t)|)$$
Where:
- $|Y(f, t)|$ is the noisy signal magnitude spectrum.
- $\mu(f)$ is the estimated noise magnitude spectrum profile.
- $\alpha \ge 1$ is the over-subtraction factor (removes music noise/artifacts).
- $\beta \ge 0$ is the spectral floor (avoids absolute zero values, reducing musical noise).

We estimate the noise profile $\mu(f)$ by averaging the magnitude spectrum of the first 0.5 seconds of the signal (which we know is silence/noise-only).

In [ ]:
# Determine number of frames corresponding to the first 0.5 seconds of noise
noise_frames = int(librosa.time_to_frames(0.5, sr=fs, hop_length=hop_length))
noise_profile = np.mean(S_noisy_mag[:, :noise_frames], axis=1, keepdims=True)

# Parameters for spectral subtraction
alpha = 2.0  # Over-subtraction
beta = 0.05  # Spectral floor

# Subtract magnitude spectrum
S_clean_est_mag = np.maximum(S_noisy_mag - alpha * noise_profile, beta * S_noisy_mag)

# Reconstruct the complex spectrogram using the estimated magnitude and noisy phase
S_denoised = S_clean_est_mag * S_noisy_phase

# Inverse STFT to reconstruct the time-domain signal
denoised_signal = librosa.istft(S_denoised, hop_length=hop_length, win_length=win_length, center=False)

# Truncate time vector to match reconstructed length if necessary
t_denoised = t[:len(denoised_signal)]

# Plot waveforms and spectrograms
plt.figure(figsize=(12, 6))
plt.subplot(2, 2, 1)
plt.plot(t, noisy_signal, color='r', alpha=0.5)
plt.title("Noisy Signal Waveform")
plt.grid(True)

plt.subplot(2, 2, 2)
plt.plot(t_denoised, denoised_signal, color='b', alpha=0.7)
plt.title("Denoised Signal Waveform")
plt.grid(True)

plt.subplot(2, 2, 3)
S_noisy_db = librosa.amplitude_to_db(S_noisy_mag, ref=np.max)
librosa.display.specshow(S_noisy_db, sr=fs, hop_length=hop_length, x_axis='time', y_axis='linear')
plt.title("Noisy Spectrogram")
plt.colorbar(format='%+2.0f dB')

plt.subplot(2, 2, 4)
S_denoised_db = librosa.amplitude_to_db(S_clean_est_mag, ref=np.max)
librosa.display.specshow(S_denoised_db, sr=fs, hop_length=hop_length, x_axis='time', y_axis='linear')
plt.title("Denoised Spectrogram")
plt.colorbar(format='%+2.0f dB')

plt.tight_layout()
plt.show()

# Play denoised audio
Audio(denoised_signal, rate=fs)

## 3. Mel-Frequency Cepstral Coefficients (MFCCs)
**MFCCs** represent the spectral envelope of a short frame of speech. They map the linear frequency spectrum to a non-linear Mel scale, which mimics human auditory system's pitch resolution (sensitive to changes at lower frequencies, less sensitive at higher frequencies).

### MFCC Extraction Pipeline:
1. Frame the signal and apply a window (e.g., Hamming).
2. Compute the Fourier Transform.
3. Apply the Mel-filterbank (a series of bandpass filters spaced linearly at low frequencies, logarithmically at high frequencies).
4. Take the logarithm of the Mel powers.
5. Apply the Discrete Cosine Transform (DCT) to decorrelate the Mel-log energies, keeping the first 12-20 coefficients.

Let's extract and plot MFCCs using Librosa.

In [ ]:
# Extract MFCCs (using 13 coefficients)
mfccs = librosa.feature.mfcc(y=denoised_signal, sr=fs, n_mfcc=13, n_fft=n_fft, hop_length=hop_length)

print(f"MFCC Matrix shape: {mfccs.shape} (13 coefficients x {mfccs.shape[1]} frames)")

# Plot MFCCs
plt.figure(figsize=(10, 4))
librosa.display.specshow(mfccs, sr=fs, hop_length=hop_length, x_axis='time')
plt.title("Mel-Frequency Cepstral Coefficients (MFCCs)")
plt.ylabel("MFCC Coefficients")
plt.colorbar()
plt.tight_layout()
plt.show()

## 4. Chroma Feature Extraction
**Chroma** features project the spectrum onto the 12 semitones of the musical octave (C, C#, D, D#, E, F, F#, G, G#, A, A#, B). Pitch profile features are extremely powerful for musical analysis, chord detection, and key estimation because they are invariant to octave differences.

Let's extract and plot Chroma features.

In [ ]:
# Extract Chroma features
chroma = librosa.feature.chroma_stft(y=denoised_signal, sr=fs, n_fft=n_fft, hop_length=hop_length)

print(f"Chroma Matrix shape: {chroma.shape} (12 semitones x {chroma.shape[1]} frames)")

# Plot Chroma
plt.figure(figsize=(10, 4))
librosa.display.specshow(chroma, sr=fs, hop_length=hop_length, x_axis='time', y_axis='chroma')
plt.title("Chroma Spectrogram")
plt.ylabel("Semitone Pitch Class")
plt.colorbar()
plt.tight_layout()
plt.show()

## 5. Audio Transformations: Pitch Shifting and Time Stretching
Audio effects allow us to manipulate signals without losing their structural characteristics:
- **Pitch Shifting**: Changes the musical pitch (frequency) of the signal without affecting its duration.
- **Time Stretching**: Speeds up or slows down the duration of the audio without changing its pitch.

Let's apply these effects to the denoised signal using Librosa's advanced DSP functions.

In [ ]:
# 1. Pitch Shift up by 4 semitones (e.g. major third interval)
pitched_signal = librosa.effects.pitch_shift(y=denoised_signal, sr=fs, n_steps=4)

# 2. Time Stretch by 1.5x (making it 50% faster)
stretched_signal = librosa.effects.time_stretch(y=denoised_signal, rate=1.5)

print(f"Original duration: {len(denoised_signal)/fs:.2f} s")
print(f"Pitched duration: {len(pitched_signal)/fs:.2f} s")
print(f"Stretched duration: {len(stretched_signal)/fs:.2f} s")

# Plot the time domain comparison
plt.figure(figsize=(12, 6))
plt.subplot(3, 1, 1)
plt.plot(denoised_signal, color='b')
plt.title("Denoised Signal (Original)")
plt.grid(True)

plt.subplot(3, 1, 2)
plt.plot(pitched_signal, color='m')
plt.title("Pitch Shifted (+4 Semitones)")
plt.grid(True)

plt.subplot(3, 1, 3)
plt.plot(stretched_signal, color='c')
plt.title("Time Stretched (1.5x speed)")
plt.grid(True)
plt.tight_layout()
plt.show()

# Listen to Pitched Audio
print("Pitch Shifted audio:")
display(Audio(pitched_signal, rate=fs))

# Listen to Stretched Audio
print("Time Stretched audio:")
display(Audio(stretched_signal, rate=fs))

## ✅ Reflection & Exercises
- **Denoising Artifacts:** Notice the tiny spikes or background "musical noise" in the denoised signal. How does increasing/decreasing $\alpha$ (over-subtraction) and $\beta$ (spectral floor) affect this musical noise?
- **Feature Space:** Look at the MFCC representation. Why is the 0th coefficient often discarded or normalized out in classification tasks?